# CV面试知识卡片 09: PyTorch CNN 与视觉模型

本节涵盖 CNN 基础、经典架构 (ResNet, VGG, MobileNet)、预训练模型、迁移学习和数据集加载。
这是计算机视觉面试的核心内容, 要求对网络结构和训练流程有深入理解。

---
## 一、选择题部分

In [ ]:
# 题目 1: nn.Conv2d 的参数
print("""
Q: nn.Conv2d(in_channels=3, out_channels=64, kernel_size=3, stride=1, padding=1)
   中 out_channels=64 的含义是什么?
A) 输入图像有 64 个通道
B) 该卷积层使用 64 个卷积核, 输出 64 个特征图
C) 卷积核的大小是 64x64
D) 该层的输出空间分辨率为 64x64
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: nn.Conv2d 关键参数:
- in_channels: 输入通道数 (RGB=3, 灰度=1, 上一层输出通道数)
- out_channels: 卷积核数量 = 输出通道数, 每个 kernel 产生一个特征图
- kernel_size: 卷积核空间尺寸 (3 表示 3x3)
- stride: 步幅, 控制输出空间下采样率
- padding: 填充, 常用 padding=kernel_size//2 保持空间尺寸
参数量 = in_channels * kernel_size^2 * out_channels + out_channels (bias)""")

In [ ]:
# 题目 2: 卷积层输出尺寸计算
print("""
Q: 输入特征图 32x32, kernel_size=5, stride=1, padding=0, 输出尺寸是?
A) 32x32
B) 28x28
C) 27x27
D) 30x30
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: 卷积输出尺寸公式:
H_out = floor((H_in + 2*padding - kernel_size) / stride) + 1
= floor((32 + 2*0 - 5) / 1) + 1
= floor(27) + 1 = 28
所以输出为 28x28。

常见组合:
- kernel=3, stride=1, padding=1: 输出尺寸不变 (same padding)
- kernel=3, stride=2, padding=1: 输出尺寸减半
- kernel=7, stride=2, padding=3: 输出尺寸减半 (ResNet 首层)""")

In [ ]:
# 题目 3: 池化层的作用
print("""
Q: 以下哪个不是池化层 (Pooling) 的作用?
A) 降低空间维度, 减少计算量
B) 增大感受野
C) 提供一定程度的平移不变性
D) 增加模型参数量
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "D"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: 池化层:
- A) 正确: MaxPool2d(2,2) 将特征图空间尺寸减半, 减少后续计算量
- B) 正确: 池化相当于在更大范围内选择/平均, 增大了等效感受野
- C) 正确: 局部区域的 max/avg 使得微小平移对输出影响较小
- D) 错误: 池化层没有可学习参数! MaxPool/AvgPool 都是固定操作
注意: 全局平均池化 (AdaptiveAvgPool2d(1)) 在 ResNet 等网络中常用,
替代全连接层, 大幅减少参数量。""")

In [ ]:
# 题目 4: 1x1 卷积的作用
print("""
Q: 1x1 卷积 (pointwise convolution) 的主要作用不包括?
A) 升维或降维通道数
B) 跨通道信息融合
C) 增大空间感受野
D) 等价于对每个空间位置做全连接
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "C"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: 1x1 卷积:
- A) 正确: 如 Conv2d(256, 64, 1) 将 256 通道降为 64 通道
- B) 正确: 1x1 卷积核覆盖所有输入通道, 实现跨通道信息交互
- C) 错误: 1x1 卷积核的空间尺寸为 1x1, 不改变空间感受野!
  感受野不变, 它只在通道维度操作
- D) 正确: 对每个像素位置, 1x1 卷积就是 C_in -> C_out 的线性变换
1x1 卷积在 Network in Network、ResNet bottleneck、MobileNet 中广泛使用。""")

In [ ]:
# 题目 5: 感受野的计算
print("""
Q: 三层 3x3 卷积 (stride=1, padding=1 堆叠) 的等效感受野是?
A) 3x3
B) 5x5
C) 7x7
D) 9x9
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "C"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: 感受野计算公式:
RF_i = RF_{i-1} + (k_i - 1) * product(strides[:i])

初始: RF_0 = 1 (单个像素)
第1层 (k=3, s=1): RF_1 = 1 + (3-1)*1 = 3
第2层 (k=3, s=1): RF_2 = 3 + (3-1)*1 = 5
第3层 (k=3, s=1): RF_3 = 5 + (3-1)*1 = 7

所以三层 3x3 卷积堆叠的感受野是 7x7, 等效于一个 7x7 卷积。
但参数量: 3*(3*3*C*C) = 27C^2 vs 7*7*C*C = 49C^2, 更少参数!
这也是 VGG 用小卷积核堆叠替代大卷积核的设计思想。""")

In [ ]:
# 题目 6: ResNet 的核心创新
print("""
Q: ResNet 的残差连接 (skip connection) 解决了什么问题?
A) 计算速度太慢的问题
B) 深层网络的退化问题 (更深的网络反而精度下降)
C) 内存不足的问题
D) 过拟合问题
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: ResNet 核心思想:
- 退化问题: 网络加深后, 训练误差反而上升 (不是过拟合, 训练误差也高)
- 残差学习: 让网络学习 F(x) = H(x) - x (残差), 而非直接学习 H(x)
  理想情况: 如果恒等映射最优, F(x)=0 比 H(x)=x 更容易学习
- skip connection: y = F(x) + x, 梯度可以直接通过快捷路径回传
  缓解了梯度消失, 使训练超深层网络 (50/101/152层) 成为可能
- ResNet-18/34 使用 BasicBlock (两个 3x3)
- ResNet-50/101/152 使用 Bottleneck (1x1-3x3-1x1)""")

In [ ]:
# 题目 7: BatchNorm 的作用
print("""
Q: 关于 BatchNorm2d, 以下说法错误的是?
A) 训练时使用当前 mini-batch 的均值和方差进行归一化
B) 推理时使用训练过程中累积的 running_mean 和 running_var
C) BatchNorm 层没有可学习参数
D) BatchNorm 有正则化效果, 因为每个 batch 的统计量有噪声
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "C"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: BatchNorm2d(num_features) 有两组可学习参数:
- gamma (weight): 缩放参数, 初始化为 1
- beta (bias): 平移参数, 初始化为 0
输出 = gamma * (x - mean) / sqrt(var + eps) + beta

A) 正确: 训练时用当前 batch 统计量
B) 正确: 推理时用指数移动平均累积的 running_mean/var
C) 错误: BN 有 2*num_features 个可学习参数 (gamma, beta)
D) 正确: batch 统计量的随机性相当于给训练添加噪声, 有正则化效果
注意: model.eval() 会将 BN 切换到推理模式。""")

In [ ]:
# 题目 8: Dropout 的原理
print("""
Q: Dropout(p=0.5) 在训练和推理时的行为分别是?
A) 训练时随机置零 50% 神经元, 推理时也随机置零 50%
B) 训练时随机置零 50% 神经元, 推理时所有神经元输出乘以 0.5
C) 训练时随机置零 50% 神经元并缩放, 推理时正常使用全部神经元
D) 训练时输出乘以 0.5, 推理时置零 50%
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "C"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: Dropout 工作机制:
- 训练时: 以概率 p 随机将神经元输出置为 0, 其余输出乘以 1/(1-p) (inverted dropout)
  缩放的目的是保证训练和推理时数值期望一致
- 推理时 (model.eval()): 不做 dropout, 使用全部神经元, 无需缩放
PyTorch 使用 inverted dropout, 即训练时已经做了缩放, 推理时直接使用即可。
常见参数: CNN 中 Dropout 较少使用 (BN 有正则化效果),
全连接层常用 p=0.5, 注意力机制中常用 Dropout(p=0.1)。""")

In [ ]:
# 题目 9: Dataset 和 DataLoader
print("""
Q: 自定义 torch.utils.data.Dataset 子类必须实现哪些方法?
A) __init__, __len__, __getitem__
B) __init__, __len__, __getitem__, __next__
C) __init__, __iter__, __next__
D) __init__, forward, backward
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "A"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: 自定义 Dataset 必须实现:
- __init__(self, ...): 初始化, 加载数据路径/标签等元信息
- __len__(self): 返回数据集大小
- __getitem__(self, idx): 根据索引返回一个样本 (和标签)

DataLoader 负责批量加载、打乱、多进程:
- batch_size: 每批样本数
- shuffle: 是否打乱 (训练时 True)
- num_workers: 数据加载的子进程数
- collate_fn: 自定义样本合并方式
- pin_memory: 为 GPU 传输预分配内存""")

In [ ]:
# 题目 10: DataLoader 的 num_workers
print("""
Q: DataLoader 中 num_workers 参数的作用是?
A) 限制 GPU 使用的线程数
B) 使用多个子进程并行加载数据, 加速数据读取
C) 指定数据集被分成多少份
D) 设置 batch 的大小
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: num_workers 控制数据加载的并行度:
- num_workers=0: 主进程加载, 简单但可能成为瓶颈
- num_workers=N: 使用 N 个子进程预取数据, GPU 训练时数据已在内存中等待
推荐设置:
- 一般设为 4~8, 或 CPU 核心数的一半
- Windows 上可能需要较小值, 避免多进程问题
- 过大可能导致内存不足和进程切换开销
注意: num_workers > 0 时, 代码需要放在 if __name__ == '__main__' 中 (尤其 Windows)""")

In [ ]:
# 题目 11: 迁移学习冻层
print("""
Q: 迁移学习中冻结特征提取层的方法是?
A) 将模型参数的 requires_grad 设为 False
B) 将模型参数的值设为 0
C) 删除这些层的参数
D) 使用更小的学习率
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "A"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: 冻层方法:
# 方法1: 冻结指定层
for param in model.features.parameters():
    param.requires_grad = False

# 方法2: 只训练特定层
for name, param in model.named_parameters():
    if 'classifier' not in name:  # 只训练分类头
        param.requires_grad = False

冻结后 optimizer 只会更新 requires_grad=True 的参数:
optimizer = SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=0.01)
优点: 减少显存占用 (不计算冻结层的梯度), 加速训练, 防止小数据集过拟合。""")

In [ ]:
# 题目 12: model.eval() vs model.train()
print("""
Q: model.eval() 会影响哪些层的行为?
A) 只影响 Dropout 层
B) 只影响 BatchNorm 层
C) 同时影响 Dropout 和 BatchNorm 层
D) 不影响任何层, 只是标记
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "C"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: model.eval() 和 model.train() 的区别:
model.train() (默认模式):
- Dropout: 随机丢弃神经元
- BatchNorm: 使用当前 batch 统计量, 并更新 running_mean/var

model.eval():
- Dropout: 不丢弃, 使用所有神经元
- BatchNorm: 使用 running_mean/var (训练时累积的全局统计量)

验证/测试时的标准写法:
model.eval()
with torch.no_grad():
    output = model(input)
model.train()  # 训练时记得切回来!
注意: eval() 只改变前向行为, no_grad() 才禁用梯度计算, 两者不同。""")

In [ ]:
# 题目 13: 分组卷积/深度可分离卷积
print("""
Q: MobileNet 的深度可分离卷积 (Depthwise Separable Conv) 将标准卷积分解为?
A) 1x1 卷积 + 3x3 卷积
B) Depthwise 卷积 + Pointwise (1x1) 卷积
C) 3x3 卷积 + 3x3 卷积
D) 5x5 卷积 + 1x1 卷积
""")

answer = "A"  # <-- 修改为你的答案

# 验证
correct = "B"
print(f"{'✓ 正确!' if answer == correct else f'✗ 错误, 正确答案是 {correct}'}")
if answer != correct:
    print("""解析: 深度可分离卷积分解:
标准卷积: Conv2d(C_in, C_out, 3)
  参数量: C_in * C_out * 3 * 3

分解为两步:
1) Depthwise: 每个输入通道独立卷积
   Conv2d(C_in, C_in, 3, groups=C_in)
   参数量: C_in * 3 * 3

2) Pointwise: 1x1 卷积做通道混合
   Conv2d(C_in, C_out, 1)
   参数量: C_in * C_out

总参数量比约 1/C_out + 1/9, 大幅减少计算量。
MobileNet 就是用这种结构实现轻量化。""")

---
## 二、编程练习部分

### 练习 1: 从零实现简单 CNN

实现 Conv -> BN -> ReLU -> Pool -> FC 的经典 CNN 结构。

In [ ]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # TODO: 定义网络结构
        # 提示:
        # self.features = nn.Sequential(
        #     nn.Conv2d(1, 32, 3, padding=1),
        #     nn.BatchNorm2d(32),
        #     nn.ReLU(),
        #     nn.MaxPool2d(2),
        #     ...
        # )
        # self.classifier = nn.Sequential(
        #     nn.Flatten(),
        #     nn.Linear(...),
        #     ...
        # )
        pass
    
    def forward(self, x):
        # TODO: 实现前向传播
        pass

# 测试
model = SimpleCNN(num_classes=10)
x = torch.randn(4, 1, 28, 28)  # batch=4, 1通道, 28x28
output = model(x)
print(f"输入: {x.shape}, 输出: {output.shape}")

In [ ]:
# ====== 参考答案 ======
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 1x28x28 -> 32x14x14
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            # Block 2: 32x14x14 -> 64x7x7
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            
            # Block 3: 64x7x7 -> 128x3x3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 测试
model = SimpleCNN(num_classes=10)
x = torch.randn(4, 1, 28, 28)
output = model(x)
print(f"输入: {x.shape}, 输出: {output.shape}")

# 统计参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数量: {total_params:,}, 可训练参数量: {trainable_params:,}")

### 练习 2: 实现残差块 (BasicBlock)

实现 ResNet 中的 BasicBlock, 包含残差连接 (skip connection)。

In [ ]:
import torch
import torch.nn as nn

class BasicBlock(nn.Module):
    """ResNet BasicBlock: 两层 3x3 卷积 + 残差连接"""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # TODO: 定义两层卷积和 shortcut
        # 提示: 如果 in_channels != out_channels 或 stride != 1,
        #       shortcut 需要用 1x1 卷积调整维度
        pass
    
    def forward(self, x):
        # TODO: 实现 y = F(x) + shortcut(x)
        pass

# 测试
block = BasicBlock(64, 64, stride=1)
x = torch.randn(2, 64, 32, 32)
out = block(x)
print(f"BasicBlock (64->64, stride=1): {x.shape} -> {out.shape}")

block2 = BasicBlock(64, 128, stride=2)
out2 = block2(x)
print(f"BasicBlock (64->128, stride=2): {x.shape} -> {out2.shape}")

In [ ]:
# ====== 参考答案 ======
import torch
import torch.nn as nn

class BasicBlock(nn.Module):
    """ResNet BasicBlock: 两层 3x3 卷积 + 残差连接"""
    expansion = 1  # BasicBlock 不扩展通道
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # 第一层卷积 (可能改变空间尺寸和通道数)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                              stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        # 第二层卷积 (不改变尺寸)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                              stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # shortcut 连接
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        
        out += identity  # 残差连接
        out = self.relu(out)
        return out

# 测试
block = BasicBlock(64, 64, stride=1)
x = torch.randn(2, 64, 32, 32)
out = block(x)
print(f"BasicBlock (64->64, stride=1): {x.shape} -> {out.shape}")

block2 = BasicBlock(64, 128, stride=2)
out2 = block2(x)
print(f"BasicBlock (64->128, stride=2): {x.shape} -> {out2.shape}")

# 用 BasicBlock 搭建一个小型 ResNet
class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)
    
    def _make_layer(self, in_ch, out_ch, num_blocks, stride):
        layers = [BasicBlock(in_ch, out_ch, stride)]
        for _ in range(1, num_blocks):
            layers.append(BasicBlock(out_ch, out_ch, stride=1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

mini_resnet = MiniResNet(num_classes=10)
x_test = torch.randn(2, 1, 28, 28)
print(f"\nMiniResNet: {x_test.shape} -> {mini_resnet(x_test).shape}")

### 练习 3: 迁移学习 — 加载预训练 ResNet, 冻结特征层, 替换分类头

使用 torchvision 的预训练 ResNet18, 冻结特征提取层, 替换最后的全连接层进行迁移学习。

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

def create_transfer_model(num_classes=5, freeze_features=True):
    """
    创建基于预训练 ResNet18 的迁移学习模型
    
    Args:
        num_classes: 目标类别数
        freeze_features: 是否冻结特征提取层
    Returns:
        model: 修改后的模型
    """
    # TODO: 加载预训练模型
    # model = models.resnet18(weights=...)
    
    # TODO: 冻结特征层
    
    # TODO: 替换分类头
    # model.fc = nn.Linear(...)
    
    pass

print("完成 TODO 后运行")

In [ ]:
# ====== 参考答案 ======
import torch
import torch.nn as nn
from torchvision import models

def create_transfer_model(num_classes=5, freeze_features=True):
    """创建基于预训练 ResNet18 的迁移学习模型"""
    # 加载预训练模型
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # 冻结特征提取层
    if freeze_features:
        for param in model.parameters():
            param.requires_grad = False
    
    # 替换分类头
    num_features = model.fc.in_features  # ResNet18 的 fc 输入维度是 512
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    )
    
    return model

# 创建模型
model = create_transfer_model(num_classes=5, freeze_features=True)

# 查看模型结构
print("分类头:")
print(model.fc)
print()

# 统计参数
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print(f"总参数量: {total_params:>12,}")
print(f"可训练参数: {trainable_params:>12,} ({trainable_params/total_params*100:.1f}%)")
print(f"冻结参数:   {frozen_params:>12,} ({frozen_params/total_params*100:.1f}%)")

# 测试前向传播
x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    output = model(x)
print(f"\n输入: {x.shape} -> 输出: {output.shape}")

### 练习 4: 自定义 Dataset 加载图片文件夹

实现一个自定义 Dataset 类, 从文件夹结构加载图片数据。
假设文件夹结构为:
```
data_root/
  train/
    cat/  (存放猫的图片)
    dog/  (存放狗的图片)
  val/
    cat/
    dog/
```

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np

class ImageFolderDataset(Dataset):
    """从文件夹结构加载图片的自定义 Dataset"""
    
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir: 数据根目录 (如 'data/train')
            transform: 图像变换操作
        """
        # TODO: 扫描文件夹, 收集图片路径和标签
        pass
    
    def __len__(self):
        # TODO: 返回数据集大小
        pass
    
    def __getitem__(self, idx):
        # TODO: 加载图片, 应用变换, 返回 (image, label) 元组
        pass

print("完成 TODO 后运行")

In [ ]:
# ====== 参考答案 ======
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np

class ImageFolderDataset(Dataset):
    """从文件夹结构加载图片的自定义 Dataset"""
    
    # 支持的图片格式
    IMG_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
    
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir: 数据根目录 (如 'data/train')
            transform: 图像变换操作
        """
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []  # (图片路径, 标签索引)
        self.class_to_idx = {}  # {类名: 索引}
        self.classes = []  # [类名]
        
        # 扫描文件夹结构
        if os.path.isdir(root_dir):
            class_names = sorted([d for d in os.listdir(root_dir) 
                                 if os.path.isdir(os.path.join(root_dir, d))])
            self.classes = class_names
            self.class_to_idx = {name: idx for idx, name in enumerate(class_names)}
            
            for class_name in class_names:
                class_dir = os.path.join(root_dir, class_name)
                label = self.class_to_idx[class_name]
                
                for fname in sorted(os.listdir(class_dir)):
                    if os.path.splitext(fname)[1].lower() in self.IMG_EXTENSIONS:
                        self.samples.append((os.path.join(class_dir, fname), label))
        
        print(f"数据集: {root_dir}")
        print(f"  类别数: {len(self.classes)}, 样本数: {len(self.samples)}")
        print(f"  类别: {self.class_to_idx}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        # 加载图片
        image = Image.open(img_path).convert('RGB')
        
        # 应用变换
        if self.transform is not None:
            image = self.transform(image)
        
        return image, label

# 用随机数据模拟测试 (不需要真实文件)
print("--- 自定义 Dataset 演示 ---")
print("实际使用方式:")
print("""
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = ImageFolderDataset('data/train', transform=train_transform)
train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=4,
    pin_memory=True,
)

# 训练循环
for images, labels in train_loader:
    # images: (B, 3, 224, 224), labels: (B,)
    outputs = model(images)
    loss = criterion(outputs, labels)
    ...
""")

### 练习 5: 计算感受野和参数量

编写工具函数计算网络各层的感受野和参数量, 用于模型分析。

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

def compute_receptive_field(layers_info):
    """
    计算各层感受野
    
    Args:
        layers_info: list of dict, 每个 dict 包含 kernel_size, stride, padding
    Returns:
        list of (layer_name, output_size, receptive_field, stride_product)
    """
    # TODO: 实现感受野计算
    pass

def count_parameters(model):
    """
    统计模型各层参数量
    Returns:
        dict: {层名: 参数量}
    """
    # TODO: 实现参数量统计
    pass

print("完成 TODO 后运行")

In [ ]:
# ====== 参考答案 ======
import torch
import torch.nn as nn
from torchvision import models

def compute_receptive_field(layers_info):
    """
    计算各层感受野
    layers_info: list of (name, kernel_size, stride)
    """
    rf = 1      # 当前感受野
    stride_prod = 1  # 累积步幅
    results = []
    
    print(f"{'层名':<30} {'卷积核':>6} {'步幅':>6} {'感受野':>8} {'累积步幅':>8}")
    print("-" * 65)
    
    for name, k, s in layers_info:
        rf = rf + (k - 1) * stride_prod
        stride_prod *= s
        results.append((name, k, s, rf, stride_prod))
        print(f"{name:<30} {k:>6} {s:>6} {rf:>8} {stride_prod:>8}")
    
    return results

def count_parameters(model):
    """统计模型各层参数量"""
    print(f"{'层名':<50} {'参数量':>12} {'形状':>20}")
    print("-" * 85)
    
    total = 0
    for name, param in model.named_parameters():
        num_params = param.numel()
        total += num_params
        if num_params > 0:
            print(f"{name:<50} {num_params:>12,} {str(list(param.shape)):>20}")
    
    print("-" * 85)
    print(f"{'总计':<50} {total:>12,}")
    return total

# 示例: VGG 风格网络的感受野
print("=== VGG 风格网络感受野 ===")
vgg_layers = [
    ("conv1_1 (3x3, s1)", 3, 1),
    ("conv1_2 (3x3, s1)", 3, 1),
    ("pool1   (2x2, s2)", 2, 2),
    ("conv2_1 (3x3, s1)", 3, 1),
    ("conv2_2 (3x3, s1)", 3, 1),
    ("pool2   (2x2, s2)", 2, 2),
    ("conv3_1 (3x3, s1)", 3, 1),
    ("conv3_2 (3x3, s1)", 3, 1),
    ("conv3_3 (3x3, s1)", 3, 1),
    ("pool3   (2x2, s2)", 2, 2),
    ("conv4_1 (3x3, s1)", 3, 1),
    ("conv4_2 (3x3, s1)", 3, 1),
    ("conv4_3 (3x3, s1)", 3, 1),
    ("pool4   (2x2, s2)", 2, 2),
    ("conv5_1 (3x3, s1)", 3, 1),
    ("conv5_2 (3x3, s1)", 3, 1),
    ("conv5_3 (3x3, s1)", 3, 1),
    ("pool5   (2x2, s2)", 2, 2),
]
compute_receptive_field(vgg_layers)

print("\n=== ResNet18 参数量统计 ===")
resnet18 = models.resnet18(weights=None)
total = count_parameters(resnet18)

# 按模块汇总
print("\n=== 按模块汇总 ===")
for name, module in resnet18.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"  {name:<15} {params:>12,} 参数 ({params/total*100:.1f}%)")